# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 9: Evaluate LLMs with (other) LLMs</font>

# <font color="#003660">Evaluate using (other) LLMs?</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how to generate test data using Argilla and HuggingFace.
        ... will know how to evaluate LLMs using <b>Larger LLMs.</b> <br>
        ... will know how to apply this using LangChain-Ollama.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [LangChain API Reference (2024)](https://python.langchain.com/api_reference/reference.html)
* [LangChain Docs (2024)](https://python.langchain.com/docs/introduction/)
* [LangChain AI (2024) Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/rewrite.ipynb?ref=blog.langchain.dev)
* [Don't Touch the Power Line](https://github.com/skaltenp/dont_touch_the_powerline)
* [Replacing Judges with Juries](https://arxiv.org/pdf/2404.18796)
* [Argilla Dataset Generator](https://huggingface.co/spaces/argilla/synthetic-data-generator)

# RAG Evaluation

![](https://github.com/olivermueller/amlta-2025/blob/main/Session_09/imgs/Pipeline.png?raw=true)

As shown by [Kaltenpoth and Müller (2024)](https://doi.org/10.1145/3717413.3717415) and [Es et al. (2024)](https://doi.org/10.18653/v1/2024.eacl-demo.16), RAG evaluation and assessment can be done using pre-defined or even LLM-generated data. These evaluations first generate questions and reference answers with an *Assessment LLM*, then answer these questions with the *Student LLM* to evaluate and finally assess the *Student LLM* answers using the *Assessment LLM*.

In [ ]:
!pip install -U pymupdf4llm datasets transformers faiss-cpu sentence-transformers accelerate langchain langchain-community langchain_openai langchain-ollama

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [ ]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull jina/jina-embeddings-v2-base-de # takes a few seconds

In [ ]:
!ollama pull gpt-oss # takes around a minute

In [ ]:
import os
import re
from tqdm.notebook import tqdm
import pymupdf4llm
import urllib

from IPython.display import display, Markdown

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

from langchain_openai import ChatOpenAI

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from typing_extensions import Annotated, TypedDict

In [ ]:
os.environ["OPENAI_API_KEY"] = ""
RETRIEVER_NAME = "jina/jina-embeddings-v2-base-de"
GENERATOR_NAME = "gpt-oss"

#  Load and Prep Documents

In [ ]:
! mkdir markdown_documents

In [ ]:
!cd markdown_documents && wget https://raw.githubusercontent.com/olivermueller/amlta-2025/refs/heads/main/Session_09/documents/Game_of_Thrones.md

In [ ]:
markdown_documents_path = "markdown_documents"

In [ ]:
def remove_markdown_links(text):
    """
    Removes Markdown links from the given text while keeping the link text.

    Args:
        text (str): The input Markdown text.

    Returns:
        str: The text with Markdown links removed.

    Yeah this was ChatGPT ;)
    """
    # Regex to match Markdown links [text](link)
    pattern = r'\[([^\]]+)\]\([^\)]+\)'
    # Replace the matched pattern with just the text inside the brackets
    cleaned_text = re.sub(pattern, r'\1', text)
    return cleaned_text

In [ ]:
markdown_file_path = "markdown_documents/Game_of_Thrones.md"

with open(markdown_file_path) as file:
    md_files = [["Game_of_Thrones.md", remove_markdown_links(file.read())]]
md_files

In [ ]:
embedding_tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v2-base-en", use_fast=False)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(embedding_tokenizer, chunk_size=512, chunk_overlap=32)
all_splits = text_splitter.create_documents(texts=[x[1] for x in md_files], metadatas=[{"source": x[0]} for x in md_files])

# Generate Questions and Reference Answers

In [ ]:
llm = ChatOpenAI(
    model = "gpt-4o",
    temperature=0,
    seed=42,
)

In [ ]:
SYSTEM_PROMPT = """
You are an assistant that generates question and reference answer based on a given document.

### Instructions:
- You will be given a document to read.
- You will then be asked to generate questions based on the document.
- You will also be asked to generate reference answers for the questions.



### Format
Document:
<here goes the document>

Generate your questions in the format:

QUESTION: <your question here>
REFERENCE ANSWER: <your reference answer here>
"""

PROMPT_TEMPLATE = """
Document:
{document}

Generate your questions in the format:

QUESTION: <your question here>
REFERENCE ANSWER: <your reference answer here>"""

In [ ]:
l = []
for split in tqdm(all_splits[:10]):
    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", PROMPT_TEMPLATE.format(document=split.page_content)),
    ]
    qa = llm.invoke(messages).content.split("QUESTION:")[-1]
    question = qa.split("REFERENCE ANSWER:")[0].strip()
    reference_answer = qa.split("REFERENCE ANSWER:")[-1].strip()
    l.append((question, reference_answer))

In [ ]:
test_set = []
for i in range(len(all_splits[:10])):
    test_set.append(
        {
            "document": all_splits[i].page_content,
            "question": l[i][0],
            "reference_answer": l[i][1],
        }
    )
    print(f"Question: {l[i][0]}")
    print(f"Reference Answer: {l[i][1]}")

# Generate Answers on the Questions with the (Smaller) LLM

In [ ]:
SYSTEM_PROMPT = """
You are an LLM that answers questions based on documents.
Answer only based on the document but not on your own knowledge.
"""

PROMPT_TEMPLATE = """
Answer the following question based on the document:
{document}

Here is the question:
{question}

Give you answer in the fomat:

ANSWER: <your answer>"""

small_llm = ChatOpenAI(
    model = "gpt-oss",
    base_url = "http://localhost:11434/v1",
    api_key="ollama",
    temperature=0,
    seed=42,
)

for i in tqdm(range(len(test_set))):
    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", PROMPT_TEMPLATE.format(document=test_set[i]["document"], question=test_set[i]["question"])),
    ]
    response = small_llm.invoke(messages).content
    test_set[i]["answer"] = response

In [ ]:
for i in range(len(all_splits[:10])):
    print(f"Question: {test_set[i]['question']}")
    print(f"Reference Answer: {test_set[i]['reference_answer']}")
    print(f"Answer: {test_set[i]['answer']}")

# Evaluate our (Small)LLM Answers with Evaluator-LLM

In [ ]:
SYSTEM_PROMPT = """
You are an assistant that validates the answers given by another LLM.

### Instructions:
- You will be given a question and an answer.
- Additionally you will be given the source document and the reference answer.
- You have to validate if the answer is correct based on the document.
- If the answer is correct, reply with "CORRECT".
- If the answer is incorrect, reply with "INCORRECT".
"""

PROMPT_TEMPLATE = """
Here is the document:
{document}

Here is the question:
{question}

Here is the reference answer:
{reference_answer}

Here is the answer to validate:
{answer}

Now give only information if the answer is correct or incorrect.

Your answer:"""

for i in tqdm(range(len(test_set[0:1]))):
    messages = [
        ("system", SYSTEM_PROMPT),
        ("human", PROMPT_TEMPLATE.format(document=test_set[i]["document"], question=test_set[i]["question"], reference_answer=test_set[i]["reference_answer"], answer=test_set[i]["answer"])),
    ]
    response = llm.invoke(messages).content
    print(test_set[i]["document"])
    print("-" * 25)
    print(test_set[i]["question"])
    print("-" * 25)
    print(test_set[i]["reference_answer"])
    print("-" * 25)
    print(test_set[i]["answer"])
    print("-" * 25)
    print(response)

## Automatic RAG Evaluation or Assessment

Automatic RAG Evaluation or Assessment usually implements an automated assessment for *Faithfulness* (Correctness), (Answer) *Relevance*, and *Context Relevance* (retrieval relevance) ([Es et al., 2024](https://doi.org/10.18653/v1/2024.eacl-demo.16); [Kaltenpoth and Müller, 2024](https://doi.org/10.1145/3717413.3717415)). While Es et al. (2024) provide their own package, we will continue with LangChain.

In [ ]:
# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = ChatOpenAI(model="gpt-4o", temperature=0, seed=42).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
            {"role": "system", "content": correctness_instructions},
            {"role": "user", "content": answers},
        ]
    )
    return grade["correct"]

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0, seed=42).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
            {"role": "system", "content": relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["relevant"]

In [ ]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[
        bool, ..., "Provide the score on if the answer hallucinates from the documents"
    ]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. You will be given FACTS and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. (2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = ChatOpenAI(model="gpt-4o", temperature=0, seed=42).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
            {"role": "system", "content": grounded_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["grounded"]

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student. Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatOpenAI(model="gpt-4o", temperature=0, seed=42).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
            {"role": "system", "content": retrieval_relevance_instructions},
            {"role": "user", "content": answer},
        ]
    )
    return grade["relevant"]

In [ ]:
fail = test_set[-1].copy()
fail["answer"] = "He hated everything about it."
test_set.append(fail)
len(test_set)

In [ ]:
import json
from types import SimpleNamespace

# Create results directory if it doesn't exist
os.makedirs("results", exist_ok=True)

# Store evaluation results
evaluation_results = []

for i in tqdm(range(len(test_set))):
    # Prepare inputs and outputs in the expected format
    inputs = {"question": test_set[i]["question"]}

    # Create a mock document object with page_content attribute
    mock_doc = SimpleNamespace(page_content=test_set[i]["document"])
    outputs = {
        "answer": test_set[i]["answer"],
        "documents": [mock_doc]
    }
    reference_outputs = {"answer": test_set[i]["reference_answer"]}

    # Evaluate all three core metrics
    try:
        corr = correctness(inputs, outputs, reference_outputs)
    except Exception as e:
        print(f"Error in correctness for item {i}: {e}")
        corr = None

    try:
        ground = groundedness(inputs, outputs)
    except Exception as e:
        print(f"Error in groundedness for item {i}: {e}")
        ground = None

    try:
        ret_rel = retrieval_relevance(inputs, outputs)
    except Exception as e:
        print(f"Error in retrieval_relevance for item {i}: {e}")
        ret_rel = None

    # Store results
    result = {
        "index": i,
        "question": test_set[i]["question"],
        "reference_answer": test_set[i]["reference_answer"],
        "answer": test_set[i]["answer"],
        "correctness": corr,
        "groundedness": ground,
        "retrieval_relevance": ret_rel
    }
    evaluation_results.append(result)

    print(f"Item {i}: Correctness={corr}, Groundedness={ground}, Retrieval Relevance={ret_rel}")

# Calculate summary statistics
summary = {
    "total_items": len(evaluation_results),
    "correctness_rate": sum(1 for r in evaluation_results if r["correctness"]) / len(evaluation_results) if evaluation_results else 0,
    "groundedness_rate": sum(1 for r in evaluation_results if r["groundedness"]) / len(evaluation_results) if evaluation_results else 0,
    "retrieval_relevance_rate": sum(1 for r in evaluation_results if r["retrieval_relevance"]) / len(evaluation_results) if evaluation_results else 0,
}

# Save results to JSON
output = {
    "summary": summary,
    "results": evaluation_results
}

with open("results/results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\n" + "="*50)
print("Evaluation Summary:")
print(f"  Correctness Rate: {summary['correctness_rate']:.2%}")
print(f"  Groundedness Rate: {summary['groundedness_rate']:.2%}")
print(f"  Retrieval Relevance Rate: {summary['retrieval_relevance_rate']:.2%}")
print(f"\nResults saved to results/results.json")

Our example is strongly simplified, if you want more evaluation frameworks:
- [LangChain Code](https://docs.langchain.com/langsmith/evaluate-rag-tutorial#reference-code).
- [RAGAs](https://docs.ragas.io/en/stable/)

But in general, is automatic evaluation super good?
It depends on the model and the task. 
This only should give you a hint how good your model is.

Combine multiple models ([Verga et al., 2024](https://arxiv.org/pdf/2404.18796)) or models and humans.

![](https://github.com/olivermueller/amlta-2025/blob/main/Session_09/imgs/judges.png?raw=true)

([Verga et al., 2024](https://arxiv.org/pdf/2404.18796))

But in general, is automatic evaluation super good?
It depends on the model and the task.
This only should give you a hint how good your model is.

Combine multiple models ([Verga et al., 2024](https://arxiv.org/pdf/2404.18796)) or models and humans.

![](https://github.com/olivermueller/amlta-2025/blob/main/Session_09/imgs/judges.png?raw=true)

([Verga et al., 2024](https://arxiv.org/pdf/2404.18796))